# Biomechanical parameters — model vs annotations

Computes ground contact time, flight time, and cadence from gait phase predictions
and compares them to manual annotation labels.

**Train set:** all optojump recordings (cross-experiment training domain)  
**Test set:** all tempos recordings, results broken down by pace (1, 2, 3)

In [22]:
import json
import os
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy import ndarray

from experiments.gait_detection.config import ExperimentConfig
from src.gait.detection.detectors import KinematicDetector, XGBoostDetector
from src.gait.detection.postprocess import (
    estimate_hmm_params,
    viterbi_decode,
)
from src.gait.gait_data.dataset import load_dataset_with_pose, train_test_split, VideoRecord
from src.gait.image.dataset import load_image_dataset
from src.gait.parameters.biomechanics import (
    gait_intervals,
    compute_cadence,
    compute_step_lengths,
    compute_joint_angles,
)

In [23]:
# ── Model to evaluate ─────────────────────────────────────────────────────────
# Valid keys: "tcn" | "img_tcn" | "resnet" | "xgb" | "img_xgb" | "kinematic" | "annotations"
MODEL = "tcn"

# paths
TCN_MANIFEST          = "data/output/gait/pose_data/stage6/manifest.json"
IMG_TCN_MANIFEST      = "data/output/gait/image/stage6/manifest.json"
RESNET_MANIFEST       = "data/output/gait/image/resnet/infer/manifest.json"
XGB_LOAO_MANIFEST     = "data/output/gait/pose_data/xgb_loao/manifest.json"
IMG_XGB_LOAO_MANIFEST = "data/output/gait/image/xgb_loao/manifest.json"
XGB_CHECKPOINT_PATH   = "data/output/gait/pose_data/stage3/checkpoints/model.pkl"
XGB_PARAMS_PATH       = "data/output/gait/pose_data/stage3/best_params.json"
IMG_XGB_MODEL_PATH    = "data/output/gait/image/stage1/checkpoints/xgboost_best.pkl"

OPTOJUMP_ANNOTATIONS_CSV = "data/output/annotations/optojump/visibility_annotations.csv"
TEMPOS_CSV               = "data/output/annotations/tempos/ml_labels.csv"
IMG_TEMPOS_FEATURES_DIR  = "data/output/gait/image_features_tempos"
RUNNER_HEIGHT_M = 2.0
POSTPROCESS_WIN = 3

In [24]:
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / "configs" / "dataset.yaml").exists():
    os.chdir(_cwd.parent)

## Load pose data + predictions

In [25]:
_manifest_idx: dict | None = None
_xgb_detector: XGBoostDetector | None = None

def _load_infer_manifest(path, label):
    """Load an inference manifest and update _manifest_idx for split lookup."""
    global _manifest_idx
    with open(path) as f:
        manifest = json.load(f)
    idx = {r["video_path"]: r for r in manifest["records"]}
    print(f"{label} manifest loaded — {len(idx)} records")
    _manifest_idx = idx
    return idx

def load_tcn_manifest():
    global _manifest_idx
    try:
        return _load_infer_manifest(TCN_MANIFEST, "TCN")
    except FileNotFoundError:
        if LABEL_SOURCE == "tcn":
            raise
        _manifest_idx = {}
        return {}

def load_img_tcn_manifest():
    return _load_infer_manifest(IMG_TCN_MANIFEST, "Image TCN")

def load_resnet_manifest():
    return _load_infer_manifest(RESNET_MANIFEST, "ResNet")

def _load_loao_manifest(path, label):
    """Load a LOAO (train-only) manifest and update _manifest_idx."""
    global _manifest_idx
    with open(path) as f:
        m = json.load(f)
    idx = {r["video_path"]: r for r in m["records"]}
    print(f"{label} LOAO manifest loaded — {len(idx)} records")
    _manifest_idx = idx
    return idx

def load_xgb_loao_manifest():
    return _load_loao_manifest(XGB_LOAO_MANIFEST, "XGBoost (pose)")

def load_img_xgb_loao_manifest():
    return _load_loao_manifest(IMG_XGB_LOAO_MANIFEST, "XGBoost (image)")


In [26]:
cfg = ExperimentConfig(annotations_csv=OPTOJUMP_ANNOTATIONS_CSV, fps=120.0, n_trim_padding=50)
cfg.annotations_csv = OPTOJUMP_ANNOTATIONS_CSV

LABEL_SOURCE = MODEL

# Train: all optojump records
pairs_train = load_dataset_with_pose(OPTOJUMP_ANNOTATIONS_CSV, fps=cfg.fps,
                                     n_trim_padding=cfg.n_trim_padding)
# Test: all tempos records
pairs_test  = load_dataset_with_pose(TEMPOS_CSV, fps=cfg.fps, dataset="tempos",
                                     n_trim_padding=cfg.n_trim_padding)

pairs = pairs_train + pairs_test
train_records = [rec for rec, _ in pairs_train]

_bio_split_map = {rec.video_path: "train" for rec, _ in pairs_train}
_bio_split_map.update({rec.video_path: "test" for rec, _ in pairs_test})

print(f"Train: {len(pairs_train)} optojump records")
print(f"Test : {len(pairs_test)} tempos records")

# Tempos image features — used as fallback for img_xgb on test records
_img_tempos_by_path: dict = {}
if os.path.exists(IMG_TEMPOS_FEATURES_DIR):
    _img_tempos = load_image_dataset(TEMPOS_CSV, IMG_TEMPOS_FEATURES_DIR,
                                     "data/input/tempos", dataset="tempos",
                                     n_trim_padding=cfg.n_trim_padding)
    _img_tempos_by_path = {r.video_path: r for r in _img_tempos}
    print(f"Tempos image features: {len(_img_tempos)} records loaded")

startprob, transmat = estimate_hmm_params([r.labels for r in train_records])
HMM_PARAMS = {"transmat": transmat, "startprob": startprob}
print(f"HMM params estimated from {len(train_records)} optojump training records")

Train: 179 optojump records
Test : 160 tempos records
Tempos image features: 160 records loaded
HMM params estimated from 179 optojump training records


## Compute per-video biomechanical parameters

In [27]:
def _probs_from_manifest(manifest_idx, rec, T):
    entry = manifest_idx.get(rec.video_path)
    if entry is None:
        return None
    probs = np.load(entry["probs_path"])
    if len(probs) != T:
        probs = probs[:T] if len(probs) > T else np.pad(probs, ((0, T - len(probs)), (0, 0)), "edge")
    return probs


def _apply_postprocess(probs):
    return viterbi_decode(probs, **HMM_PARAMS)


_tcn_manifest_idx: dict | None = None

def load_labels_tcn(rec: VideoRecord) -> ndarray | None:
    global _tcn_manifest_idx
    if _tcn_manifest_idx is None:
        _tcn_manifest_idx = load_tcn_manifest()
    probs = _probs_from_manifest(_tcn_manifest_idx, rec, len(rec.labels))
    return None if probs is None else _apply_postprocess(probs)


_img_tcn_manifest_idx: dict | None = None

def load_labels_img_tcn(rec: VideoRecord) -> ndarray | None:
    global _img_tcn_manifest_idx
    if _img_tcn_manifest_idx is None:
        _img_tcn_manifest_idx = load_img_tcn_manifest()
    probs = _probs_from_manifest(_img_tcn_manifest_idx, rec, len(rec.labels))
    return None if probs is None else _apply_postprocess(probs)


_resnet_manifest_idx: dict | None = None

def load_labels_resnet(rec: VideoRecord) -> ndarray | None:
    global _resnet_manifest_idx
    if _resnet_manifest_idx is None:
        _resnet_manifest_idx = load_resnet_manifest()
    probs = _probs_from_manifest(_resnet_manifest_idx, rec, len(rec.labels))
    return None if probs is None else _apply_postprocess(probs)


_xgb_loao_idx: dict | None = None
_xgb_clf = None
_xgb_feature_idx = None

def load_labels_xgb(rec: VideoRecord) -> ndarray | None:
    global _xgb_loao_idx, _xgb_clf, _xgb_feature_idx
    if _xgb_loao_idx is None:
        _xgb_loao_idx = load_xgb_loao_manifest()
    probs = _probs_from_manifest(_xgb_loao_idx, rec, len(rec.labels))
    if probs is not None:
        return _apply_postprocess(probs)
    # Fallback: direct inference for records not in LOAO manifest (tempos test)
    if _xgb_clf is None:
        if not os.path.exists(XGB_CHECKPOINT_PATH):
            return None
        _xgb_clf = joblib.load(XGB_CHECKPOINT_PATH)
        if os.path.exists(XGB_PARAMS_PATH):
            with open(XGB_PARAMS_PATH) as f:
                _xgb_feature_idx = json.load(f).get("feature_idx")
    feats = rec.features if _xgb_feature_idx is None else rec.features[:, _xgb_feature_idx]
    proba = _xgb_clf.predict_proba(feats.astype(np.float32))
    return _apply_postprocess(proba)


_img_xgb_loao_idx: dict | None = None
_img_xgb_clf = None

def load_labels_img_xgb(rec: VideoRecord) -> ndarray | None:
    global _img_xgb_loao_idx, _img_xgb_clf
    if _img_xgb_loao_idx is None:
        _img_xgb_loao_idx = load_img_xgb_loao_manifest()
    probs = _probs_from_manifest(_img_xgb_loao_idx, rec, len(rec.labels))
    if probs is not None:
        return _apply_postprocess(probs)
    # Fallback: direct inference using tempos image features
    img_rec = _img_tempos_by_path.get(rec.video_path)
    if img_rec is None:
        return None
    if _img_xgb_clf is None:
        if not os.path.exists(IMG_XGB_MODEL_PATH):
            return None
        _img_xgb_clf = joblib.load(IMG_XGB_MODEL_PATH)
    T = len(rec.labels)
    proba = _img_xgb_clf.predict_proba(img_rec.features.astype(np.float32))
    proba = proba[:T] if len(proba) > T else np.pad(proba, ((0, T - len(proba)), (0, 0)), "edge")
    return _apply_postprocess(proba)


def load_labels_kinematic(rec: VideoRecord) -> ndarray:
    return KinematicDetector().predict(rec.features, cfg.fps)


def load_labels_annotations(rec: VideoRecord) -> ndarray:
    return rec.labels

In [28]:
def _study_test(video_path: str) -> tuple[int, int]:
    path = Path(video_path)
    study_match = re.search(r"study_(\d+)", video_path)
    study_id = int(study_match.group(1)) if study_match else 0
    try:
        test_id = int(path.stem.split("_")[-1])
    except (ValueError, IndexError):
        test_id = 0
    return study_id, test_id

In [29]:
def make_bio_df(load_labels: callable) -> tuple[pd.DataFrame, dict]:
    results = []
    all_steps = {"contact": [], "flight": [], "length": []}
    n_skipped = 0

    for rec, pose_df in pairs:
        labels = load_labels(rec)
        if labels is None:
            n_skipped += 1
            continue

        study_id, test_id = _study_test(rec.video_path)
        meta = {"athlete": rec.athlete, "study_id": study_id, "test_id": test_id}

        ivs = gait_intervals(labels, fps=cfg.fps)
        cad = compute_cadence(labels, fps=cfg.fps)
        sl_df = compute_step_lengths(labels, pose_df, fps=cfg.fps, runner_height_m=RUNNER_HEIGHT_M)

        contact_segs = ivs["contact"][1:-1]
        flight_segs  = ivs["flight"][1:-1]
        sl_trimmed   = sl_df.iloc[1:-1]

        for key, source, col in [
            ("contact", contact_segs, "duration_s"),
            ("flight",  flight_segs,  "duration_s"),
        ]:
            step_df = pd.DataFrame(source).rename(columns={col: f"{key}_s"})
            all_steps[key].append(step_df.assign(**meta))

        if "step_length_m" in sl_trimmed.columns:
            sl_rows = sl_trimmed[["step_length_m"]].dropna()
        else:
            sl_rows = pd.DataFrame(columns=["step_length_m"])
        all_steps["length"].append(sl_rows.assign(**meta))

        gcts = [c["duration_s"] for c in contact_segs]
        flts = [f["duration_s"] for f in flight_segs]
        sls  = sl_trimmed["step_length_m"].dropna() if "step_length_m" in sl_trimmed.columns else pd.Series([], dtype=float)

        results.append({
            **meta,
            "video_path": rec.video_path,
            "mean_gct_s": np.nanmean(gcts) if gcts else np.nan,
            "mean_flight_s": np.nanmean(flts) if flts else np.nan,
            "mean_step_length_m": np.nanmean(sls) if not sls.empty else np.nan,
            "cadence_spm": cad["steps_per_minute"],
            "n_steps": cad["n_steps"],
        })

    if n_skipped:
        print(f"Skipped {n_skipped} records with no manifest entry (test-set records not in LOAO manifest).")
    return pd.DataFrame(results), all_steps


In [30]:
_loaders = {
    "tcn":         load_labels_tcn,
    "img_tcn":     load_labels_img_tcn,
    "resnet":      load_labels_resnet,
    "xgb":         load_labels_xgb,
    "img_xgb":     load_labels_img_xgb,
    "kinematic":   load_labels_kinematic,
    "annotations": load_labels_annotations,
}
if LABEL_SOURCE not in _loaders:
    raise ValueError(f"Unknown LABEL_SOURCE={LABEL_SOURCE!r}. Choose from: {list(_loaders)}")

print(f"Computing biomechanics — label source: {LABEL_SOURCE!r}")
bio_df, all_steps = make_bio_df(_loaders[LABEL_SOURCE])
contact_all  = pd.concat(all_steps["contact"], ignore_index=True)
flight_all   = pd.concat(all_steps["flight"],  ignore_index=True)
step_len_all = pd.concat(all_steps["length"],  ignore_index=True)

def _get_pace(video_path: str):
    stem = re.sub(r"\.(?:MP4|mp4)$", "", video_path.split("/")[-1], flags=re.IGNORECASE)
    if stem == "1": return 1
    if stem == "2": return 2
    if re.match(r"3", stem): return 3
    return None

bio_df["split"] = bio_df["video_path"].map(lambda vp: _bio_split_map.get(vp, "unknown"))
bio_df["pace"]  = bio_df["video_path"].apply(_get_pace)
print(f"  train: {(bio_df['split']=='train').sum()}  test: {(bio_df['split']=='test').sum()}")

Computing biomechanics — label source: 'tcn'
TCN manifest loaded — 339 records
  train: 179  test: 160


---
## Error vs annotations — all models

RMSE and bias of GCT / flight time when comparing model predictions to manual annotations.
Train = optojump (LOAO-based for xgb/img_xgb, full model for others).  
Test = tempos (direct inference).  
`RMSE / Floor` uses the 120 fps quantisation floor for a single step:
`floor = 1 / sqrt(6)` frames ≈ 0.41 frames.

In [31]:
def calculate_gait_metrics_summary(df, ground_truth_col_suffix, fps=None, quant_floor=None):
    """
    Computes a comprehensive biomechanical error summary for gait metrics.
    Temporal metrics (GCT, flight time) are reported in seconds, ms, and frames.
    """
    metrics_spec = [
        ("mean_gct_s",    "GCT",         "s"),
        ("mean_flight_s", "Flight time", "s"),
    ]

    splits = [
        ("train", df["split"] == "train"),
        ("test",  df["split"] == "test"),
        ("all",   pd.Series(True, index=df.index)),
    ]

    summary_rows = []
    for split_label, mask in splits:
        sub = df[mask]
        if sub.empty:
            continue

        for v_col, label, unit in metrics_spec:
            oj_col   = f"{v_col}_{ground_truth_col_suffix}"
            diff     = sub[v_col] - sub[oj_col]
            bias_s   = diff.mean()
            rmse_s   = np.sqrt((diff ** 2).mean())
            std_s    = diff.std()
            nrmse    = rmse_s / sub[oj_col].mean() * 100
            bias_pct = abs(bias_s) / sub[oj_col].mean() * 100

            row = {
                "Split":      split_label,
                "Metric":     f"{label} ({unit})",
                "n":          len(sub),
                "Video mean": round(sub[v_col].mean(), 4),
                "OJ mean":    round(sub[oj_col].mean(), 4),
                "Bias (s)":   f"{bias_s:+.4f}",
                "Bias (ms)":  f"{bias_s * 1000:+.2f}",
                "Bias %":     f"{bias_pct:+.1f}%",
                "RMSE (s)":   round(rmse_s, 4),
                "RMSE (ms)":  round(rmse_s * 1000, 2),
                "NRMSE":      f"{nrmse:.1f}%",
                "STD (s)":    round(std_s, 4),
            }
            if fps is not None:
                row["Bias (frames)"] = round(bias_s * fps, 2)
                row["RMSE (frames)"] = round(rmse_s * fps, 2)
                row["STD (frames)"]  = round(std_s  * fps, 2)
            summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).set_index(["Split", "Metric"])

    frames_df = None
    if fps is not None and quant_floor is not None:
        frame_rows = []
        for (split, metric), row in summary_df.iterrows():
            if not metric.endswith("(s)"):
                continue
            frame_rows.append({
                "Split":          split,
                "Metric":         metric,
                "RMSE (s)":       row["RMSE (s)"],
                "RMSE (ms)":      row["RMSE (ms)"],
                "RMSE (frames)":  row["RMSE (frames)"],
                "Floor (frames)": round(quant_floor, 2),
                "RMSE / Floor":   round(float(row["RMSE (frames)"]) / quant_floor, 1),
            })
        frames_df = pd.DataFrame(frame_rows).set_index(["Split", "Metric"])

    return summary_df, frames_df

---
## Single-model: error vs annotations (train / test)

In [32]:
quant_floor_frames = 1 / np.sqrt(6)

# Annotation biomechanics as ground truth
gt_bio_df, _ = make_bio_df(load_labels_annotations)
gt_bio_df["split"] = gt_bio_df["video_path"].map(lambda vp: _bio_split_map.get(vp, "unknown"))

# Merge on video_path for 1-to-1 alignment
bio_vs_gt = bio_df.merge(
    gt_bio_df[["video_path", "mean_gct_s", "mean_flight_s"]],
    on="video_path", how="inner", suffixes=(None, "_gt")
)

summary_df, frames_df = calculate_gait_metrics_summary(
    bio_vs_gt, "gt", fps=cfg.fps, quant_floor=quant_floor_frames
)
display(summary_df)
display(frames_df)

n  Video mean  OJ mean Bias (s) Bias (ms)  Bias %  \
Split Metric                                                                 
train GCT (s)          179      0.1275   0.1245  +0.0012     +1.24   +1.0%   
      Flight time (s)  179      0.1001   0.1023  -0.0025     -2.54   +2.5%   
test  GCT (s)          160      0.1722   0.1548  +0.0185    +18.48  +11.9%   
      Flight time (s)  160      0.1024   0.1278  -0.0257    -25.65  +20.1%   
all   GCT (s)          339      0.1490   0.1390  +0.0095     +9.55   +6.9%   
      Flight time (s)  339      0.1012   0.1144  -0.0136    -13.58  +11.9%   

                       RMSE (s)  RMSE (ms)  NRMSE  STD (s)  Bias (frames)  \
Split Metric                                                                
train GCT (s)            0.0122      12.20   9.8%   0.0122           0.15   
      Flight time (s)    0.0118      11.83  11.6%   0.0116          -0.30   
test  GCT (s)            0.1085     108.47  70.1%   0.1072           2.22   
      Flight time (s)    0.0440      44.02  34.5%   0.0359          -3.08   
all   GCT (s)            0.0758      75.80  54.5%   0.0753           1.15   
      Flight time (s)    0.0316      31.60  27.6%   0.0286          -1.63   

                       RMSE (frames)  STD (frames)  
Split Metric                                        
train GCT (s)                   1.46          1.46  
      Flight time (s)           1.42          1.39  
test  GCT (s)                  13.02         12.87  
      Flight time (s)           5.28          4.31  
all   GCT (s)                   9.10          9.04  
      Flight time (s)           3.79          3.43

RMSE (s)  RMSE (ms)  RMSE (frames)  Floor (frames)  \
Split Metric                                                                
train GCT (s)            0.0122      12.20           1.46            0.41   
      Flight time (s)    0.0118      11.83           1.42            0.41   
test  GCT (s)            0.1085     108.47          13.02            0.41   
      Flight time (s)    0.0440      44.02           5.28            0.41   
all   GCT (s)            0.0758      75.80           9.10            0.41   
      Flight time (s)    0.0316      31.60           3.79            0.41   

                       RMSE / Floor  
Split Metric                         
train GCT (s)                   3.6  
      Flight time (s)           3.5  
test  GCT (s)                  31.9  
      Flight time (s)          12.9  
all   GCT (s)                  22.3  
      Flight time (s)           9.3

---
## All-model comparison vs annotations

Runs every label source and collects error metrics vs manual annotation labels.
Train split = optojump; test split = tempos (all paces combined, then by pace).

In [33]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

_cmp_loaders = {
    "tcn":       load_labels_tcn,
    "img_tcn":   load_labels_img_tcn,
    "resnet":    load_labels_resnet,
    "xgb":       load_labels_xgb,
    "img_xgb":   load_labels_img_xgb,
    "kinematic": load_labels_kinematic,
}

quant_floor_frames = 1 / np.sqrt(6)

# Ground-truth annotation biomechanics (once for all)
print("annotations ...", end=" ", flush=True)
_gt_bio_df, _ = make_bio_df(load_labels_annotations)
_gt_bio_df["split"] = _gt_bio_df["video_path"].map(lambda vp: _bio_split_map.get(vp, "unknown"))
_gt_bio_df["pace"]  = _gt_bio_df["video_path"].apply(_get_pace)
print(f"done  ({len(_gt_bio_df)} records)")

_gt_rows = []

for _model_name, _loader in _cmp_loaders.items():
    print(f"  {_model_name} ...", end=" ", flush=True)
    try:
        _bdf, _ = make_bio_df(_loader)
    except Exception as _e:
        print(f"error: {_e}")
        continue
    if _bdf.empty:
        print("no records"); continue

    _bdf["split"] = _bdf["video_path"].map(lambda vp: _bio_split_map.get(vp, "unknown"))

    _m_gt = _bdf.merge(
        _gt_bio_df[["video_path", "mean_gct_s", "mean_flight_s"]],
        on="video_path", how="inner", suffixes=(None, "_gt")
    )
    _smry_gt, _ = calculate_gait_metrics_summary(
        _m_gt, "gt", fps=cfg.fps, quant_floor=quant_floor_frames
    )
    for (_sp, _mt), _row in _smry_gt.iterrows():
        _gt_rows.append({"Model": _model_name, "Split": _sp, "Metric": _mt,
                         **_row.to_dict()})
    print(f"done  (n={len(_m_gt)})")

cmp_gt = pd.DataFrame(_gt_rows)
_show_cols = ["n", "RMSE (s)", "RMSE (ms)", "RMSE (frames)", "NRMSE", "Bias (ms)", "Bias (frames)", "Bias %"]

print("\n=== vs Annotations ===")
for _split in ["train", "test", "all"]:
    _sub = cmp_gt[cmp_gt["Split"] == _split]
    if _sub.empty: continue
    print(f"\n── Split: {_split} ──")
    print(_sub.set_index(["Model", "Metric"])[_show_cols])

# ── Test set grouped by pace ──────────────────────────────────────────────────
# print("\n=== Test set — by pace ===")
# _pace_rows = []
# for _model_name, _loader in _cmp_loaders.items():
#     try:
#         _bdf, _ = make_bio_df(_loader)
#     except Exception:
#         continue
#     if _bdf.empty: continue
#     _bdf["split"] = _bdf["video_path"].map(lambda vp: _bio_split_map.get(vp, "unknown"))
#     _bdf["pace"]  = _bdf["video_path"].apply(_get_pace)
#     _test = _bdf[_bdf["split"] == "test"]
#     if _test.empty: continue
#
#     _m_gt = _test.merge(
#         _gt_bio_df[["video_path", "mean_gct_s", "mean_flight_s"]],
#         on="video_path", how="inner", suffixes=(None, "_gt")
#     )
#     for _pace in [1, 2, 3]:
#         _g = _m_gt[_m_gt["pace"] == _pace]
#         if _g.empty: continue
#         _gct_err  = (_g["mean_gct_s"]    - _g["mean_gct_s_gt"])
#         _flt_err  = (_g["mean_flight_s"] - _g["mean_flight_s_gt"])
#         _pace_rows.append({
#             "Model":              _model_name,
#             "Pace":               _pace,
#             "n":                  len(_g),
#             "GCT pred (ms)":      round(_g["mean_gct_s"].mean()    * 1000, 1),
#             "GCT annot (ms)":     round(_g["mean_gct_s_gt"].mean() * 1000, 1),
#             "GCT RMSE (ms)":      round(np.sqrt((_gct_err ** 2).mean()) * 1000, 2),
#             "GCT RMSE (fr)":      round(np.sqrt((_gct_err ** 2).mean()) * cfg.fps, 2),
#             "Flight pred (ms)":   round(_g["mean_flight_s"].mean()    * 1000, 1),
#             "Flight annot (ms)":  round(_g["mean_flight_s_gt"].mean() * 1000, 1),
#             "Flight RMSE (ms)":   round(np.sqrt((_flt_err ** 2).mean()) * 1000, 2),
#             "Flight RMSE (fr)":   round(np.sqrt((_flt_err ** 2).mean()) * cfg.fps, 2),
#         })
#
# pace_df = pd.DataFrame(_pace_rows)
# if not pace_df.empty:
#     for _model_name in _cmp_loaders:
#         _sub = pace_df[pace_df["Model"] == _model_name].set_index("Pace")
#         if _sub.empty: continue
#         print(f"\n── {_model_name} ──")
#         display(_sub.drop(columns="Model"))


annotations ... done  (339 records)
  tcn ... done  (n=339)
  img_tcn ... Image TCN manifest loaded — 339 records
done  (n=339)
  resnet ... ResNet manifest loaded — 339 records
done  (n=339)
  xgb ... XGBoost (pose) LOAO manifest loaded — 179 records
done  (n=339)
  img_xgb ... XGBoost (image) LOAO manifest loaded — 179 records
done  (n=339)
  kinematic ... done  (n=339)

=== vs Annotations ===

── Split: train ──
                             n  RMSE (s)  RMSE (ms)  RMSE (frames)  NRMSE  \
Model     Metric                                                            
tcn       GCT (s)          179    0.0122      12.20           1.46   9.8%   
          Flight time (s)  179    0.0118      11.83           1.42  11.6%   
img_tcn   GCT (s)          179    0.0110      11.01           1.32   8.8%   
          Flight time (s)  179    0.0109      10.86           1.30  10.6%   
resnet    GCT (s)          179    0.0107      10.69           1.28   8.6%   
          Flight time (s)  179    0.0084  